In [2]:
import numpy as np

# ==========================================
# 1. PROBLEM AND ENVIRONMENT DEFINITION
# ==========================================
start_point = np.array([0.0, 0.0])  # P_0
goal_point = np.array([10.0, 10.0])  # P_{M+1}
M = 4  # Number of intermediate waypoints
MAP_BOUNDS = (0.0, 10.0)  # [Min, Max] for X and Y

# Circular obstacles: {'center': (x, y), 'radius': r}
obstacles = [
    {'center': np.array([3.0, 3.0]), 'radius': 1.5},
    {'center': np.array([6.5, 6.0]), 'radius': 1.8},
]

# ==========================================
# 2. USER HYPERPARAMETER INPUT
# ==========================================
print('--- Configure PSO Hyperparameters ---')
try:
  num_particles = int(
      input('Enter Swarm Size / Number of Particles (e.g., 40): ') or '40'
  )
  max_iter = int(input('Enter Maximum Iterations (e.g., 150): ') or '150')
  w = float(input('Enter Inertia Weight w (e.g., 0.5): ') or '0.5')
  c1 = float(input('Enter Cognitive Coefficient c1 (e.g., 1.5): ') or '1.5')
  c2 = float(input('Enter Social Coefficient c2 (e.g., 2.0): ') or '2.0')
except ValueError:
  print('Invalid input detected. Falling back to default values.')
  num_particles, max_iter, w, c1, c2 = 40, 150, 0.5, 1.5, 2.0

lambda_penalty = 5000.0  # Massive penalty multiplier for obstacle breach


# ==========================================
# 3. ROBUST COST FUNCTION (SEGMENT CHECKING)
# ==========================================
def compute_total_cost(waypoints):
  """Calculates path length and heavily penalizes line segments that intersect obstacles."""
  # Reconstruct full ordered path sequence: [P_0, P_1, ..., P_M, P_{M+1}]
  path = np.vstack([start_point, waypoints.reshape(M, 2), goal_point])

  # 1. Calculate total Euclidean path length
  segments = np.diff(path, axis=0)
  path_length = np.sum(np.linalg.norm(segments, axis=1))

  # 2. Calculate line segment obstacle penalties
  total_penalty = 0.0

  # Loop through every line segment connecting the waypoints
  for i in range(len(path) - 1):
    p1 = path[i]
    p2 = path[i + 1]

    # Sample 15 distinct points along this specific line segment
    for t in np.linspace(0, 1, 15):
      sampled_point = p1 + t * (p2 - p1)

      # Check if this sampled point violates any obstacle boundaries
      for obs in obstacles:
        center = obs['center']
        radius = obs['radius']

        distance = np.linalg.norm(sampled_point - center)
        # 0.2 represents a small safety buffer around the obstacle
        if distance < (radius + 0.2):
          total_penalty += (radius + 0.2) - distance

  return path_length + (lambda_penalty * total_penalty)


# ==========================================
# 4. SWARM INITIALIZATION
# ==========================================
positions = np.random.uniform(
    low=MAP_BOUNDS[0], high=MAP_BOUNDS[1], size=(num_particles, M, 2)
)
velocities = np.zeros((num_particles, M, 2))

pbest_positions = np.copy(positions)
pbest_costs = np.array([compute_total_cost(p) for p in positions])

gbest_idx = np.argmin(pbest_costs)
gbest_position = np.copy(pbest_positions[gbest_idx])
gbest_cost = pbest_costs[gbest_idx]

# ==========================================
# 5. MAIN PSO OPTIMIZATION LOOP
# ==========================================
for iteration in range(max_iter):
  for i in range(num_particles):
    r1, r2 = np.random.rand(), np.random.rand()

    # Update Velocity vector
    velocities[i] = (
        w * velocities[i]
        + c1 * r1 * (pbest_positions[i] - positions[i])
        + c2 * r2 * (gbest_position - positions[i])
    )

    # Update Position coordinates
    positions[i] += velocities[i]

    # Enforce map limits
    positions[i] = np.clip(positions[i], MAP_BOUNDS[0], MAP_BOUNDS[1])

    # Evaluate fitness
    current_cost = compute_total_cost(positions[i])

    # Personal best check
    if current_cost < pbest_costs[i]:
      pbest_costs[i] = current_cost
      pbest_positions[i] = np.copy(positions[i])

      # Global best check
      if current_cost < gbest_cost:
        gbest_cost = current_cost
        gbest_position = np.copy(positions[i])

# ==========================================
# 6. OUTPUT RESULTS
# ==========================================
optimal_path = np.vstack([start_point, gbest_position.reshape(M, 2), goal_point])
print('\n=== Optimization Complete ===')
print(f'Final Shortest Safe Path Cost: {gbest_cost:.4f}')
print('Generated Coordinates [X, Y] along the path:')
for idx, pt in enumerate(optimal_path):
  if idx == 0:
    print(f'  Start   : [{pt[0]:.2f}, {pt[1]:.2f}]')
  elif idx == len(optimal_path) - 1:
    print(f'  Goal    : [{pt[0]:.2f}, {pt[1]:.2f}]')
  else:
    print(f'  Waypt {idx}: [{pt[0]:.2f}, {pt[1]:.2f}]')


--- Configure PSO Hyperparameters ---
Enter Swarm Size / Number of Particles (e.g., 40): 50
Enter Maximum Iterations (e.g., 150): 200
Enter Inertia Weight w (e.g., 0.5): 0.5
Enter Cognitive Coefficient c1 (e.g., 1.5): 1.5
Enter Social Coefficient c2 (e.g., 2.0): 2.0

=== Optimization Complete ===
Final Shortest Safe Path Cost: 14.7895
Generated Coordinates [X, Y] along the path:
  Start   : [0.00, 0.00]
  Waypt 1: [1.33, 3.74]
  Waypt 2: [2.71, 5.11]
  Waypt 3: [5.33, 7.65]
  Waypt 4: [10.00, 10.00]
  Goal    : [10.00, 10.00]
